In [11]:
import pandas as pd
import json
import ast

# Load the main task data
df = pd.read_csv("workflows_objects_task.csv", encoding='utf-8-sig')

# Drop unnecessary columns
df.drop(columns=[
    "kind", "schema", "hash", "triggers", "events",
    "actions", "confirmed_by", "for_each", "rework_upon"
], inplace=True)

# Ensure DataFrame format
df = pd.DataFrame(df)

# --- Step 1: Parse 'attachments' safely ---
def parse_attachments(val):
    try:
        return json.loads(val) if isinstance(val, str) else []
    except json.JSONDecodeError:
        return []

df['parsed_attachments'] = df['attachments'].apply(parse_attachments)

# --- Step 2: Extract URLs and Titles ---
def extract_urls_titles(attachments):
    urls = []
    titles = []
    for item in attachments:
        if isinstance(item, dict):
            urls.append(item.get('url', ''))
            titles.append(item.get('title', ''))
    return urls, titles

df['urls'], df['titles'] = zip(*df['parsed_attachments'].apply(extract_urls_titles))

# --- Step 3: Add dynamic columns for URLs and Titles ---
max_count = df['urls'].apply(len).max()

for i in range(1, max_count + 1):
    df[f'Url-{i}'] = df['urls'].apply(lambda x: x[i - 1] if i <= len(x) else '')
    df[f'Title-{i}'] = df['titles'].apply(lambda x: x[i - 1] if i <= len(x) else '')

# Cleanup intermediate columns
df.drop(columns=['parsed_attachments', 'urls', 'titles'], inplace=True)

# Fill NA values
df.fillna('NA', inplace=True)

# Print relevant columns (optional)
print(df[['id'] + [col for col in df.columns if col.startswith('Url-') or col.startswith('Title-')]])

         id                                              Url-1  \
0     11404  https://zeiss.sharepoint.com/:x:/r/sites/05449...   
1     11404  https://zeiss.sharepoint.com/:x:/r/sites/05449...   
2      1649                                                      
3       178  https://zeiss.sharepoint.com/:p:/r/sites/05449...   
4       170                                                      
...     ...                                                ...   
9987   1232                                                      
9988      5  https://zeiss.sharepoint.com/:p:/r/sites/05449...   
9989      5  https://zeiss.sharepoint.com/:p:/r/sites/05449...   
9990    162                                                      
9991   3242  https://zeiss.sharepoint.com/:w:/r/sites/05449...   

                              Title-1  \
0                        Rework Tasks   
1                        Rework Tasks   
2                                       
3     9814 - Column Bakeout procedure   
4 

In [3]:
df.columns

Index(['id', 'name', 'description', 'attachments', 'Url-1', 'Title-1', 'Url-2',
       'Title-2', 'Url-3', 'Title-3', 'Url-4', 'Title-4', 'Url-5', 'Title-5',
       'Url-6', 'Title-6', 'Url-7', 'Title-7', 'Url-8', 'Title-8', 'Url-9',
       'Title-9', 'Url-10', 'Title-10', 'Url-11', 'Title-11', 'Url-12',
       'Title-12', 'Url-13', 'Title-13', 'Url-14', 'Title-14', 'Url-15',
       'Title-15', 'Url-16', 'Title-16', 'Url-17', 'Title-17'],
      dtype='object')

In [4]:
df.to_excel('cleaned_workflow_objects_task.xlsx')

In [1]:
import pandas as pd
df1 = pd.read_excel('workflows_objects_takt.xlsx')

In [2]:
df1

,hash,kind,schema,id,name,description,attachments,triggers,events,sap_operation,goods_in,workitem_order,limitted_to,for_each_unit_of,rework_upon
0,<memory at 0x0000000021A79480>,section,takt,1,Pre Assembly,NaN,[],NaN,[],3100.0,0.0,"[1, 2, 3, 4]",[],NaN,[]
1,<memory at 0x0000000021A790C0>,section,takt,5,Takt 4 Assembly,NaN,[],NaN,[],3400.0,1.0,"[49, 50, 51, 52, 53, 54, 55, 56, 1543, 57, 154...",[],NaN,[]
2,<memory at 0x0000000021A79540>,section,takt,6,Takt 5 Assembly,NaN,[],NaN,[],3500.0,1.0,"[59, 1545, 1546, 1547, 1548, 60, 61, 62, 63, 6...",[],NaN,[]
3,<memory at 0x0000000021A796C0>,section,takt,3,Takt 2 Assembly,NaN,[],NaN,[],3200.0,1.0,"[24, 25, 26, 27, 28, 29, 30, 31]",[],NaN,[]
4,<memory at 0x0000000021A79780>,section,takt,4,Takt 3 Assembly,NaN,[],NaN,[],3300.0,1.0,"[32, 33, 34, 35, 36, 37, 38, 39, 40, 41, 42, 4...",[],NaN,[]
...,...,...,...,...,...,...,...,...,...,...,...,...,...,...,...
2975,<memory at 0x00000000221FBAC0>,section,takt,4,Takt 3 Assembly,NaN,[],NaN,"[{'event': 'completed', 'actions': ['activateN...",3300.0,NaN,"[11127, 747, 748, 32, 33, 34, 35, 11389, 668, ...",[],NaN,[]
2976,<memory at 0x00000000221FBB80>,section,takt,2,Takt 1 Assembly,NaN,[],NaN,[],3100.0,NaN,"[11103, 11104, 743, 10679, 11253, 934, 10, 12,...",[],NaN,[]
2977,<memory at 0x00000000221FBC40>,section,takt,2,Takt 1 Assembly,NaN,[],NaN,[],3100.0,NaN,"[11103, 11104, 743, 10679, 11253, 934, 10, 12,...",[],NaN,[]
2978,<memory at 0x00000000221FBD00>,section,takt,2,Takt 1 Assembly,NaN,[],NaN,[],3100.0,NaN,"[11103, 11104, 743, 10679, 11253, 934, 10, 12,...",[],NaN,[]


In [3]:
df1.columns

Index(['hash', 'kind', 'schema', 'id', 'name', 'description', 'attachments',
       'triggers', 'events', 'sap_operation', 'goods_in', 'workitem_order',
       'limitted_to', 'for_each_unit_of', 'rework_upon'],
      dtype='object')

In [4]:
df1.drop(columns=[
    'hash', 'kind', 'schema', 'description', 'attachments',
    'triggers', 'events', 'sap_operation', 'goods_in',
    'limitted_to', 'for_each_unit_of', 'rework_upon'
], inplace=True)

In [5]:
df1

,id,name,workitem_order
0,1,Pre Assembly,"[1, 2, 3, 4]"
1,5,Takt 4 Assembly,"[49, 50, 51, 52, 53, 54, 55, 56, 1543, 57, 154..."
2,6,Takt 5 Assembly,"[59, 1545, 1546, 1547, 1548, 60, 61, 62, 63, 6..."
3,3,Takt 2 Assembly,"[24, 25, 26, 27, 28, 29, 30, 31]"
4,4,Takt 3 Assembly,"[32, 33, 34, 35, 36, 37, 38, 39, 40, 41, 42, 4..."
...,...,...,...
2975,4,Takt 3 Assembly,"[11127, 747, 748, 32, 33, 34, 35, 11389, 668, ..."
2976,2,Takt 1 Assembly,"[11103, 11104, 743, 10679, 11253, 934, 10, 12,..."
2977,2,Takt 1 Assembly,"[11103, 11104, 743, 10679, 11253, 934, 10, 12,..."
2978,2,Takt 1 Assembly,"[11103, 11104, 743, 10679, 11253, 934, 10, 12,..."


In [6]:
rename_dict = {
    'id': 'sub_id',
    'name': 'Takt Name',
    'workitem_order': 'id'
}

df1.rename(columns=rename_dict, inplace = True)

In [7]:
df1.columns

Index(['sub_id', 'Takt Name', 'id'], dtype='object')

In [8]:
# Prepare the expanded rows
expanded_rows = []

for _, row in df1.iterrows():
    sub_id = row['sub_id']
    takt_name = row['Takt Name']
    ids_str = row['id']
    # Remove curly braces and any spacing, then split
    ids = [x.strip() for x in ids_str.strip('{}').split(',')]
    for single_id in ids:
        expanded_rows.append({
            'sub_id': sub_id,
            'Takt Name': takt_name,
            'id': single_id
        })

In [9]:
df1= pd.DataFrame(expanded_rows, columns=['sub_id', 'Takt Name', 'id'])
df1.to_excel('cleaned_workflows_objects_takt.xlsx', index=True)

In [10]:
df1

,sub_id,Takt Name,id
0,1,Pre Assembly,[1
1,1,Pre Assembly,2
2,1,Pre Assembly,3
3,1,Pre Assembly,4]
4,5,Takt 4 Assembly,[49
...,...,...,...
55818,2,Takt 1 Assembly,20
55819,2,Takt 1 Assembly,21
55820,2,Takt 1 Assembly,22
55821,2,Takt 1 Assembly,23


In [97]:
df1.dtypes

sub_id        int64
Takt Name    object
id           object
dtype: object

In [98]:
# def clean_and_convert_id(val):
#     if pd.isnull(val):
#         return None  # or you can use 0, or pd.NA if you want
#     val_str = str(val).replace('[', '').replace(']', '').strip()
#     return int(val_str)

# df1.columns = [col.strip() for col in df1.columns]  # Clean up column names

# df1['id'] = df1['id'].apply(clean_and_convert_id).astype('Int64')  # Convert to standard int

# df1.to_excel('cleaned_workflows_objects_takt.xlsx', index=False)
# df1


def clean_and_convert_id(val):
    if pd.isnull(val):
        return None
    val_str = str(val).replace('[','').replace(']','').strip()
    if val_str == '':
        return None
    try:
        return int(val_str)
    except ValueError:
        return None  # or raise an error/log it depending on your use case

df1.columns = [col.strip() for col in df1.columns]

df1['id'] = df1['id'].apply(clean_and_convert_id)
df1 = df1[~df1.duplicated(subset=['sub_id', 'Takt Name', 'id'], keep='first')]
df1['id'] = df1['id'].fillna(0).astype(int)
df1.to_excel('cleaned_workflows_objects_takt.xlsx', index=True)

In [99]:
df1

,sub_id,Takt Name,id
0,1,Pre Assembly,1
1,1,Pre Assembly,2
2,1,Pre Assembly,3
3,1,Pre Assembly,4
4,5,Takt 4 Assembly,49
...,...,...,...
54217,5,Takt 4 Assembly,1261
55319,4,Takt 3 Assembly,1262
55715,2,Takt 1 Assembly,1263
55730,2,Takt 1 Assembly,1248


In [ ]:
import pandas as pd
df1 = pd.read_excel('workflows_objects_takt.xlsx')

df1.drop(columns=[
    'hash', 'kind', 'schema', 'description', 'attachments',
    'triggers', 'events', 'sap_operation', 'goods_in',
    'limitted_to', 'for_each_unit_of', 'rework_upon'
], inplace=True)

rename_dict = {
    'id': 'sub_id',
    'name': 'Takt Name',
    'workitem_order': 'id'
}

df1.rename(columns=rename_dict, inplace = True)

# Prepare the expanded rows
expanded_rows = []

for _, row in df1.iterrows():
    sub_id = row['sub_id']
    takt_name = row['Takt Name']
    ids_str = row['id']
    # Remove curly braces and any spacing, then split
    ids = [x.strip() for x in ids_str.strip('{}').split(',')]
    for single_id in ids:
        expanded_rows.append({
            'sub_id': sub_id,
            'Takt Name': takt_name,
            'id': single_id
        })

df1= pd.DataFrame(expanded_rows, columns=['sub_id', 'Takt Name', 'id'])
df1.to_excel('cleaned_workflows_objects_takt.xlsx', index=True)

def clean_and_convert_id(val):
    if pd.isnull(val):
        return None
    val_str = str(val).replace('[','').replace(']','').strip()
    if val_str == '':
        return None
    try:
        return int(val_str)
    except ValueError:
        return None  # or raise an error/log it depending on your use case

df1.columns = [col.strip() for col in df1.columns]

df1['id'] = df1['id'].apply(clean_and_convert_id)
df1 = df1[~df1.duplicated(subset=['sub_id', 'Takt Name', 'id'], keep='first')]
df1['id'] = df1['id'].fillna(0).astype(int)
df1.to_excel('cleaned_workflows_objects_takt.xlsx', index=True)

In [7]:
df1

,hash,kind,schema,id,name,description,attachments,triggers,events,sap_operation,goods_in,workitem_order,limitted_to,for_each_unit_of,rework_upon
0,<memory at 0x0000000021A79480>,section,takt,1,Pre Assembly,NaN,[],NaN,[],3100.0,0.0,"[1, 2, 3, 4]",[],NaN,[]
1,<memory at 0x0000000021A790C0>,section,takt,5,Takt 4 Assembly,NaN,[],NaN,[],3400.0,1.0,"[49, 50, 51, 52, 53, 54, 55, 56, 1543, 57, 154...",[],NaN,[]
2,<memory at 0x0000000021A79540>,section,takt,6,Takt 5 Assembly,NaN,[],NaN,[],3500.0,1.0,"[59, 1545, 1546, 1547, 1548, 60, 61, 62, 63, 6...",[],NaN,[]
3,<memory at 0x0000000021A796C0>,section,takt,3,Takt 2 Assembly,NaN,[],NaN,[],3200.0,1.0,"[24, 25, 26, 27, 28, 29, 30, 31]",[],NaN,[]
4,<memory at 0x0000000021A79780>,section,takt,4,Takt 3 Assembly,NaN,[],NaN,[],3300.0,1.0,"[32, 33, 34, 35, 36, 37, 38, 39, 40, 41, 42, 4...",[],NaN,[]
...,...,...,...,...,...,...,...,...,...,...,...,...,...,...,...
2975,<memory at 0x00000000221FBAC0>,section,takt,4,Takt 3 Assembly,NaN,[],NaN,"[{'event': 'completed', 'actions': ['activateN...",3300.0,NaN,"[11127, 747, 748, 32, 33, 34, 35, 11389, 668, ...",[],NaN,[]
2976,<memory at 0x00000000221FBB80>,section,takt,2,Takt 1 Assembly,NaN,[],NaN,[],3100.0,NaN,"[11103, 11104, 743, 10679, 11253, 934, 10, 12,...",[],NaN,[]
2977,<memory at 0x00000000221FBC40>,section,takt,2,Takt 1 Assembly,NaN,[],NaN,[],3100.0,NaN,"[11103, 11104, 743, 10679, 11253, 934, 10, 12,...",[],NaN,[]
2978,<memory at 0x00000000221FBD00>,section,takt,2,Takt 1 Assembly,NaN,[],NaN,[],3100.0,NaN,"[11103, 11104, 743, 10679, 11253, 934, 10, 12,...",[],NaN,[]


In [11]:
# Load secondary workflow data for mapping
df1 = pd.read_excel('workflows_objects_takt.xlsx')

# Drop unnecessary columns from df1
df1.drop(columns=[
    'hash', 'kind', 'schema', 'description', 'attachments',
    'triggers', 'events', 'sap_operation', 'goods_in',
    'limitted_to', 'for_each_unit_of', 'rework_upon'
], inplace=True)

# Convert stringified sets/lists in workitem_order to Python lists
df1['workitem_order'] = df1['workitem_order'].apply(
    lambda x: ast.literal_eval(x) if isinstance(x, str) else x
)

# Build mapping from workitem_order values to df1 ids
workitem_to_df1id = {}

for _, row in df1.iterrows():
    df1_id = row['id']
    for wid in row['workitem_order']:
        if wid in workitem_to_df1id:
            workitem_to_df1id[wid].append(df1_id)
        else:
            workitem_to_df1id[wid] = [df1_id]

# Map df["id"] to df1 using the mapping
df['mapped_df1_id'] = df['id'].apply(lambda x: workitem_to_df1id.get(x, [None])[0])

# Remove exact duplicate rows (keeps the first occurrence)
#df = df.drop_duplicates()

df.fillna(' ', inplace=True)

# # Save final output
df.to_excel('freshclean_workflows_objects_task2.xlsx', index=False)

C:\Users\y7rbasav\AppData\Local\Temp\ipykernel_31392\125591458.py:33: FutureWarning: Setting an item of incompatible dtype is deprecated and will raise an error in a future version of pandas. Value ' ' has dtype incompatible with float64, please explicitly cast to a compatible dtype first.
  df.fillna(' ', inplace=True)


In [12]:
df

,id,name,description,attachments,Url-1,Title-1,Url-2,Title-2,Url-3,Title-3,...,Title-13,Url-14,Title-14,Url-15,Title-15,Url-16,Title-16,Url-17,Title-17,mapped_df1_id
0,11404,Rework Tasks /Tests to be repeated,After a part change / column intervention plea...,[{'url': 'https://zeiss.sharepoint.com/:x:/r/s...,https://zeiss.sharepoint.com/:x:/r/sites/05449...,Rework Tasks,,,,,...,,,,,,,,,,681.0
2,1649,Final Gun Vaccum - VP,NA,[],,,,,,,...,,,,,,,,,,14.0
3,178,When pumped to spec open up manual valves and CIV,Start when system vacuum is ≤ 2.0e-5 mbar,[{'url': 'https://zeiss.sharepoint.com/:p:/r/s...,https://zeiss.sharepoint.com/:p:/r/sites/05449...,9814 - Column Bakeout procedure,,,,,...,,,,,,,,,,14.0
4,170,New Task,NA,[],,,,,,,...,,,,,,,,,,13.0
5,187,New Task,NA,[],,,,,,,...,,,,,,,,,,21.0
...,...,...,...,...,...,...,...,...,...,...,...,...,...,...,...,...,...,...,...,...,...
9982,1291,Install Emergency Off Circuit,NA,[],,,,,,,...,,,,,,,,,,285.0
9983,6931,Stage Drift Program,Please ensure a gold on carbon sample is used ...,[{'url': 'https://zeiss.sharepoint.com/:w:/r/s...,https://zeiss.sharepoint.com/:w:/r/sites/05449...,Stage Drift WI,https://zeiss.sharepoint.com/sites/05449/Volum...,Stage Drift Test Program,,,...,,,,,,,,,,382.0
9985,921,Water Flow Tests,If the water temperature constantly reads 26 d...,[],,,,,,,...,,,,,,,,,,6.0
9990,162,Denka Tip Shim Re-Measurement,Maximum allowed Excess Tip Gap = 0.02mm,[],,,,,,,...,,,,,,,,,,12.0


In [16]:
# 1. Explode takts so each row is a single (workflow_takts_id, workflow_takts_name, workitem_id)
exploded = df1.explode('workitem_order')
exploded = exploded.rename(columns={
    'id': 'sub_id',
    'name': 'Takt Name',
    'workitem_order': 'id'
})

# 2. Merge with tasks to get task name, if available
mapped = exploded.merge(
    df1.columns,
    on='workitem_id',
    how='left'  # left join: keep all workitem_ids even if not matched in tasks
)

# 3. Reorder and show result
result = mapped[['workitem_id', 'workitem_name', 'workflow_takts_id', 'workflow_takts_name']]

print(result)

       workitem_id                                      workitem_name  \
0                1                                       Pre Assembly   
1                1                                       Pre Assembly   
2                2                                    Takt 1 Assembly   
3                2                                    Takt 1 Assembly   
4                2                                    Takt 1 Assembly   
...            ...                                                ...   
215808          22                           Gun Conditioning & Runup   
215809          23  Takt 5 - ESB Conditioning (10 Mins Setup + 12H...   
215810          23   ESB Conditioning (10 Mins Setup + 12Hr Run Time)   
215811          23                                  ESB Conditioning    
215812        1248                                                NaN   

        workflow_takts_id workflow_takts_name  
0                       1        Pre Assembly  
1                       1  

In [21]:
result.to_excel('freshclean_workflows_objects_task2.xlsx', index=False)

In [17]:
result.drop_duplicates()

,workitem_id,workitem_name,workflow_takts_id,workflow_takts_name
0,1,Pre Assembly,1,Pre Assembly
2,2,Takt 1 Assembly,1,Pre Assembly
54,3,Takt 2 Assembly,1,Pre Assembly
67,4,Takt 3 Assembly,1,Pre Assembly
123,49,New Takt,5,Takt 4 Assembly
...,...,...,...,...
199837,1261,NaN,5,Takt 4 Assembly
209441,1262,NaN,4,Takt 3 Assembly
212433,1263,NaN,2,Takt 1 Assembly
212762,1248,NaN,2,Takt 1 Assembly


In [19]:
result.to_excel('2.xlsx', index=False)

In [20]:
import pandas as pd
import json
import ast

# Load both CSV files
df_task = pd.read_csv("workflows_objects_task.csv", encoding='utf-8-sig')
df_takt = pd.read_csv("workflows_objects_takt.csv", encoding='utf-8-sig')

# Safely parse stringified Python lists in workitem_order column
df_takt['workitem_order'] = df_takt['workitem_order'].apply(
    lambda x: ast.literal_eval(x) if isinstance(x, str) else x
)

# Build a mapping from workitem_id to parent takt id and name
workitem_to_takt = {}
for _, row in df_takt.iterrows():
    mapped_id = row['id']
    mapped_name = row['name']
    for wid in row['workitem_order']:
        workitem_to_takt[wid] = {'mapped_takt_id': mapped_id, 'mapped_takt_name': mapped_name}

# Optional: Build a mapping from workitem_id to task name (for use later)
taskid_to_name = dict(zip(df_task['id'], df_task['name']))

# Build "exploded" rows: each (task id, action text)
exploded_rows = []
for _, row in df_task.iterrows():
    task_id = row['id']
    task_name = row['name']

    # Try to parse actions field
    try:
        actions = json.loads(row['actions']) if isinstance(row['actions'], str) else []
    except Exception:
        actions = []

    # If actions is a list, extract each action's text
    for action in actions:
        if isinstance(action, dict):
            text = action.get('text', '')
            exploded_rows.append({
                "task_id": task_id,
                "task_name": task_name,
                "text": text
            })

# Build DataFrame
df_exploded = pd.DataFrame(exploded_rows)

# Map each task id to the corresponding takt info (id and name)
df_exploded['mapped_takt_id'] = df_exploded['task_id'].map(
    lambda tid: workitem_to_takt.get(tid, {}).get('mapped_takt_id')
)
df_exploded['mapped_takt_name'] = df_exploded['task_id'].map(
    lambda tid: workitem_to_takt.get(tid, {}).get('mapped_takt_name')
)

# Arrange columns as desired
df_result = df_exploded[['mapped_takt_id', 'mapped_takt_name', 'task_id', 'task_name', 'text']]

print(df_result)

# Optionally, save to CSV
# df_result.to_csv('tasks_with_mapped_takt.csv', index=False)

       mapped_takt_id                       mapped_takt_name  task_id  \
0               681.0              Additional  Documentation    11404   
1                14.0  TAKT 6 Final Column Assembly & Checks     1649   
2                14.0  TAKT 6 Final Column Assembly & Checks     1649   
3                40.0                            Accessories      186   
4                40.0                            Accessories      186   
...               ...                                    ...      ...   
21769           285.0                 Services and PC Config     3242   
21770           285.0                 Services and PC Config     3242   
21771           285.0                 Services and PC Config     3242   
21772           285.0                 Services and PC Config     3242   
21773           285.0                 Services and PC Config     3242   

                                task_name  \
0      Rework Tasks /Tests to be repeated   
1                   Final Gun Vac